<a href="https://colab.research.google.com/github/roywang01-boop/AAI2026/blob/main/Excercise%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
react_prompt = """
You are a Python assistant using a ReACT loop.

Follow this exact structure:

Reason: (max 6 bullet points)
Act: Provide Python code in ONE code block
Observe: Explain what the code output should be
Fix: Mention at least 2 edge cases and how your code handles them

Task:
Write Python that summarizes transactions into:
- total_spend (sum of positive amounts)
- total_refunds (absolute sum of negative amounts)
- net (sum of all amounts)
- by_category sorted desc by spend
- top_merchants sorted desc by spend

Constraints:
- Standard library only
- Handle empty input
- Raise a helpful error if amount is not numeric
- Round currency to 2 decimals
"""
print(react_prompt)


You are a Python assistant using a ReACT loop.

Follow this exact structure:

Reason: (max 6 bullet points)
Act: Provide Python code in ONE code block
Observe: Explain what the code output should be
Fix: Mention at least 2 edge cases and how your code handles them

Task:
Write Python that summarizes transactions into:
- total_spend (sum of positive amounts)
- total_refunds (absolute sum of negative amounts)
- net (sum of all amounts)
- by_category sorted desc by spend
- top_merchants sorted desc by spend

Constraints:
- Standard library only
- Handle empty input
- Raise a helpful error if amount is not numeric
- Round currency to 2 decimals



In [3]:
model_response = """
Reason:
- Define a Transaction structure to store inputs cleanly
- Loop through transactions and track totals
- Separate positive spend from negative refunds
- Aggregate spending by category and merchant
- Sort category and merchant totals descending
- Return a dictionary with rounded values

Act:
from dataclasses import dataclass
from typing import List, Dict, Tuple

@dataclass
class Transaction:
    date: str
    merchant: str
    category: str
    amount: float

def summarize_spend(transactions: List[Transaction]) -> Dict[str, object]:
    if not transactions:
        return {
            "total_spend": 0.0,
            "total_refunds": 0.0,
            "net": 0.0,
            "by_category": [],
            "top_merchants": [],
        }

    spend_by_category: Dict[str, float] = {}
    spend_by_merchant: Dict[str, float] = {}
    total_spend = 0.0
    total_refunds = 0.0
    net = 0.0

    for t in transactions:
        if not isinstance(t.amount, (int, float)):
            raise TypeError(f"amount must be numeric, got {type(t.amount)} for {t}")
        net += float(t.amount)

        if t.amount > 0:
            total_spend += t.amount
            spend_by_category[t.category] = spend_by_category.get(t.category, 0.0) + t.amount
            spend_by_merchant[t.merchant] = spend_by_merchant.get(t.merchant, 0.0) + t.amount
        elif t.amount < 0:
            total_refunds += abs(t.amount)

    by_category = sorted(spend_by_category.items(), key=lambda x: x[1], reverse=True)
    top_merchants = sorted(spend_by_merchant.items(), key=lambda x: x[1], reverse=True)

    return {
        "total_spend": round(total_spend, 2),
        "total_refunds": round(total_refunds, 2),
        "net": round(net, 2),
        "by_category": [(k, round(v, 2)) for k, v in by_category],
        "top_merchants": [(k, round(v, 2)) for k, v in top_merchants],
    }

Observe:
- The function returns totals and sorted breakdowns.
- Refunds appear in total_refunds but do not count toward category spend.

Fix:
- Empty list returns zeros and empty lists.
- Non-numeric amount raises a clear TypeError.
"""

In [4]:
code_text = model_response.split("Act:")[1].split("Observe:")[0].strip()
exec(code_text)

print("Loaded summarize_spend() and Transaction ✅")

Loaded summarize_spend() and Transaction ✅


In [5]:
sample = [
    Transaction("2026-02-01", "Target", "Groceries", 54.23),
    Transaction("2026-02-02", "Netflix", "Entertainment", 15.49),
    Transaction("2026-02-03", "Trader Joe's", "Groceries", 31.10),
    Transaction("2026-02-04", "Uber", "Transport", 22.35),
    Transaction("2026-02-05", "Target", "Groceries", -10.00),  # refund
    Transaction("2026-02-06", "Chipotle", "Dining", 12.67),
    Transaction("2026-02-06", "Chipotle", "Dining", 9.13),
]

result = summarize_spend(sample)
print(result)

{'total_spend': 144.97, 'total_refunds': 10.0, 'net': 134.97, 'by_category': [('Groceries', 85.33), ('Transport', 22.35), ('Dining', 21.8), ('Entertainment', 15.49)], 'top_merchants': [('Target', 54.23), ("Trader Joe's", 31.1), ('Uber', 22.35), ('Chipotle', 21.8), ('Netflix', 15.49)]}


In [6]:
print("Empty input test:")
print(summarize_spend([]))

Empty input test:
{'total_spend': 0.0, 'total_refunds': 0.0, 'net': 0.0, 'by_category': [], 'top_merchants': []}


In [7]:
print("Non-numeric amount test (should raise TypeError):")
bad = [Transaction("2026-02-01", "Test", "Other", "not_a_number")]

try:
    summarize_spend(bad)
except Exception as e:
    print("Caught error:", repr(e))

Non-numeric amount test (should raise TypeError):
Caught error: TypeError("amount must be numeric, got <class 'str'> for Transaction(date='2026-02-01', merchant='Test', category='Other', amount='not_a_number')")
